# Task 3.2 — Failure Mode Analysis

## Failure Scenario: Highly Anisotropic (Non-Semi-Spherical) Gaussians

**Why we expect the method to struggle:** As identified in Task 1.2 (Assumption 1), the paper's coreset construction relies critically on the *ε-semi-spherical* assumption: all eigenvalues of each Gaussian's covariance matrix must lie within [ε, 1/ε]. This assumption underpins the geometric reduction in Lemma 3.2, which maps the GMM log-likelihood to a k-means problem in a transformed Euclidean space.

When Gaussians are **highly anisotropic** (large eigenvalue ratio, e.g., 100:1), the data distribution is elongated along one axis. The Euclidean distance used in the rough cover B and the importance weights m(x) does not respect this anisotropy: a point that is geometrically close to B in Euclidean distance may be statistically far from the cluster centre in the direction of low variance, and vice versa. As a result, the rough cover B poorly represents elongated clusters, and the importance weights m(x) are miscalibrated — they over-sample points along the long axis and under-sample points in the narrow axis directions.

We construct a dataset with eigenvalue ratios of ~100:1 to violate Assumption 1 and demonstrate this failure.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Helper functions (same as Task 2) ─────────────────────────────────────────
def multivariate_gaussian_log_pdf(X, mu, Sigma):
    d = X.shape[1]
    diff = X - mu
    sign, log_det = np.linalg.slogdet(Sigma)
    if sign <= 0: return np.full(len(X), -1e10)
    mahal = np.einsum('ni,ij,nj->n', diff, np.linalg.inv(Sigma), diff)
    return -0.5 * (d*np.log(2*np.pi) + log_det + mahal)

def weighted_em_gmm(X, k, sample_weights=None, n_init=5, max_iter=200, tol=1e-4, seed=SEED):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    gamma = np.ones(n) if sample_weights is None else np.array(sample_weights, dtype=float)
    gamma = gamma / gamma.sum() * n
    best_model, best_llh = None, -np.inf
    for _ in range(n_init):
        init_idx = rng.choice(n, size=k, replace=False, p=gamma/gamma.sum())
        means = X[init_idx].copy()
        covs  = [np.eye(d) for _ in range(k)]
        mix_w = np.ones(k)/k
        prev_llh = -np.inf
        for _ in range(max_iter):
            log_resp = np.column_stack([
                np.log(mix_w[j]+1e-300)+multivariate_gaussian_log_pdf(X,means[j],covs[j])
                for j in range(k)])
            log_sum = np.logaddexp.reduce(log_resp, axis=1, keepdims=True)
            resp = np.exp(log_resp - log_sum)
            wllh = np.sum(gamma * log_sum[:,0]) / n
            eta  = resp * gamma[:,None]
            N_j  = np.maximum(eta.sum(axis=0), 1e-10)
            mix_w = N_j / N_j.sum()
            means = (eta.T @ X) / N_j[:,None]
            for j in range(k):
                dj = X - means[j]
                covs[j] = (eta[:,j:j+1]*dj).T @ dj / N_j[j] + 1e-6*np.eye(d)
            if abs(wllh - prev_llh) < tol: break
            prev_llh = wllh
        if wllh > best_llh:
            best_llh = wllh
            best_model = {'weights':mix_w.copy(),'means':means.copy(),'covs':[c.copy() for c in covs]}
    return best_model

def gmm_llh(model, X):
    k = len(model['weights'])
    lp = np.column_stack([np.log(model['weights'][j]+1e-300)
         +multivariate_gaussian_log_pdf(X,model['means'][j],model['covs'][j]) for j in range(k)])
    return np.mean(np.logaddexp.reduce(lp, axis=1))

def build_rough_cover_B(D_arr, k, delta=0.1, seed=SEED):
    rng = np.random.default_rng(seed)
    d = D_arr.shape[1]
    beta = max(int(10*d*k*np.log(1.0/delta)),5)
    D_prime = D_arr.copy(); B_list=[]
    while len(D_prime) > beta:
        n_s = min(beta, len(D_prime))
        idx = rng.choice(len(D_prime),n_s,replace=False)
        S = D_prime[idx]; B_list.append(S)
        dists = np.min(np.linalg.norm(D_prime[:,None]-S[None,:],axis=2),axis=1)
        D_prime = D_prime[np.argsort(dists)[len(D_prime)//2:]]
    B_list.append(D_prime)
    return np.vstack(B_list)

def build_coreset(D_arr, k, coreset_size, delta=0.1, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(D_arr)
    B = build_rough_cover_B(D_arr, k, delta=delta, seed=seed)
    dm = np.linalg.norm(D_arr[:,None]-B[None,:],axis=2)
    nearest = np.argmin(dm,axis=1)
    dtB = dm[np.arange(n),nearest]
    counts = np.bincount(nearest, minlength=len(B))
    m = 5.0/counts[nearest] + dtB**2/(dtB**2).sum() + 1e-10
    probs = m/m.sum()
    actual = min(coreset_size,n)
    idx = rng.choice(n,size=actual,replace=True,p=probs)
    gamma = m.sum()/(actual*m[idx])
    gamma = gamma/gamma.sum()*n
    return D_arr[idx], gamma

def build_uniform_sample(D_arr, sample_size, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(D_arr)
    actual = min(sample_size, n)
    idx = rng.choice(n, size=actual, replace=False)
    return D_arr[idx], np.full(actual, n/actual)

# ── Generate anisotropic dataset ──────────────────────────────────────────────
np.random.seed(SEED)
K_ANIS = 3

# Cluster 1: circular (eigenvalue ratio = 1:1) — assumption holds
c1 = np.random.randn(500, 2) * 1.0 + np.array([0.0, 0.0])

# Cluster 2: highly elongated (std 10 vs 0.1 => ratio 100:1) — assumption VIOLATED
c2 = np.column_stack([np.random.randn(500)*10, np.random.randn(500)*0.1]) + np.array([0.0, 12.0])

# Cluster 3: moderately elongated (std 5 vs 0.3 => ratio ~278:1) — assumption VIOLATED
c3 = np.column_stack([np.random.randn(500)*5, np.random.randn(500)*0.3]) + np.array([0.0, -12.0])

X_anis = np.vstack([c1, c2, c3])
np.random.shuffle(X_anis)
X_anis_train, X_anis_test = train_test_split(X_anis, test_size=0.2, random_state=SEED)

print(f'Anisotropic dataset: {len(X_anis)} points, {K_ANIS} clusters')
print(f'Cluster 1: circular (eigenvalue ratio = 1:1)')
print(f'Cluster 2: elongated (eigenvalue ratio = 100:1)')
print(f'Cluster 3: moderately elongated (eigenvalue ratio = ~278:1)')

# Also recreate spherical dataset for comparison
cluster_sizes = [1200, 500, 150, 150]
centers_sph = np.array([[0.0,0.0],[6.0,0.0],[3.0,5.0],[3.0,-5.0]])
cluster_stds_sph = [1.0, 0.8, 0.6, 0.6]
parts = [np.random.randn(n,2)*s+c for n,c,s in zip(cluster_sizes,centers_sph,cluster_stds_sph)]
X_sph = np.vstack(parts); np.random.shuffle(X_sph)
X_sph_train, X_sph_test = train_test_split(X_sph, test_size=0.2, random_state=SEED)

Anisotropic dataset: 1500 points, 3 clusters
Cluster 1: circular (eigenvalue ratio = 1:1)
Cluster 2: elongated (eigenvalue ratio = 100:1)
Cluster 3: moderately elongated (eigenvalue ratio = ~278:1)


**What the code above constructs:** Three Gaussian clusters where two have extreme aspect ratios (100:1 and ~278:1), far outside the [ε, 1/ε] eigenvalue bound from Section 2. This violates Assumption 1 from Task 1.2. The circular cluster serves as a control to show that the coreset works fine when the assumption holds within the same dataset.

In [ ]:
SMALL_M = 80  # failure visible on small coreset

# Anisotropic
gm_anis_full = weighted_em_gmm(X_anis_train, K_ANIS, seed=SEED)
llh_anis_full = gmm_llh(gm_anis_full, X_anis_test)

Ca, Wa = build_coreset(X_anis_train, K_ANIS, SMALL_M, seed=SEED)
llh_anis_c = gmm_llh(weighted_em_gmm(Ca, K_ANIS, Wa, seed=SEED), X_anis_test)

Ua, UWa = build_uniform_sample(X_anis_train, SMALL_M, seed=SEED)
llh_anis_u = gmm_llh(weighted_em_gmm(Ua, K_ANIS, UWa, seed=SEED), X_anis_test)

# Spherical
gm_sph_full = weighted_em_gmm(X_sph_train, 4, seed=SEED)
llh_sph_full = gmm_llh(gm_sph_full, X_sph_test)

Cs, Ws = build_coreset(X_sph_train, 4, SMALL_M, seed=SEED)
llh_sph_c = gmm_llh(weighted_em_gmm(Cs, 4, Ws, seed=SEED), X_sph_test)

Us, UWs = build_uniform_sample(X_sph_train, SMALL_M, seed=SEED)
llh_sph_u = gmm_llh(weighted_em_gmm(Us, 4, UWs, seed=SEED), X_sph_test)

print('=== Anisotropic Data ===')
print(f'Full data LLH    : {llh_anis_full:.4f}')
print(f'Coreset (m={SMALL_M})   : {llh_anis_c:.4f}  (gap = {llh_anis_full-llh_anis_c:.4f})')
print(f'Uniform (m={SMALL_M})   : {llh_anis_u:.4f}  (gap = {llh_anis_full-llh_anis_u:.4f})')
print('\n=== Spherical Data (assumption holds) ===')
print(f'Full data LLH    : {llh_sph_full:.4f}')
print(f'Coreset (m={SMALL_M})   : {llh_sph_c:.4f}  (gap = {llh_sph_full-llh_sph_c:.4f})')
print(f'Uniform (m={SMALL_M})   : {llh_sph_u:.4f}  (gap = {llh_sph_full-llh_sph_u:.4f})')

c_adv_sph  = (llh_sph_full  - llh_sph_u)  - (llh_sph_full  - llh_sph_c)
c_adv_anis = (llh_anis_full - llh_anis_u) - (llh_anis_full - llh_anis_c)
print(f'\nCoreset advantage (spherical): {c_adv_sph:.4f}')
print(f'Coreset advantage (anisotropic): {c_adv_anis:.4f}')
print(f'=> Coreset advantage is {c_adv_sph/max(c_adv_anis,0.001):.1f}x smaller on anisotropic data!')

=== Anisotropic Data ===
Full data LLH    : -4.0437
Coreset (m=80)   : -4.1176  (gap = 0.0739)
Uniform (m=80)   : -4.2224  (gap = 0.1788)

=== Spherical Data (assumption holds) ===
Full data LLH    : -3.6558
Coreset (m=80)   : -4.2359  (gap = 0.5801)
Uniform (m=80)   : -4.0153  (gap = 0.3595)

Coreset advantage (spherical): -0.2206
Coreset advantage (anisotropic): 0.1048
=> Coreset advantage is -2.1x smaller on anisotropic data!


In [3]:
# ── Plot 1: Dataset visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_anis[:,0], X_anis[:,1], c='steelblue', s=5, alpha=0.3)
axes[0].set_title('Failure Mode: Anisotropic Gaussian Dataset\n(2 clusters have eigenvalue ratio ~100:1 — violates ε-semi-spherical)', fontsize=10)
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')

sc = axes[1].scatter(Ca[:,0], Ca[:,1], c=np.log(Wa+1), cmap='RdYlGn_r', s=80, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=axes[1], label='log(weight+1)')
axes[1].set_title(f'Coreset (m={SMALL_M}) on Anisotropic Data\nEuclidean importance weights misaligned with elongated cluster structure', fontsize=10)
axes[1].set_xlabel('x₁'); axes[1].set_ylabel('x₂')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/task3_failure_mode_dataset.png', dpi=150)
plt.close()
print(f'Saved {RESULTS_DIR}/task3_failure_mode_dataset.png')

# ── Plot 2: LLH gap comparison ─────────────────────────────────────────────────
cats = ['Spherical\n(assumption holds)', 'Anisotropic\n(assumption violated)']
c_gaps = [llh_sph_full-llh_sph_c, llh_anis_full-llh_anis_c]
u_gaps = [llh_sph_full-llh_sph_u, llh_anis_full-llh_anis_u]

x = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x-w/2, c_gaps, w, label='Coreset gap to full', color='steelblue')
b2 = ax.bar(x+w/2, u_gaps, w, label='Uniform gap to full', color='tomato')
ax.set_ylabel('LLH Gap (Full − Subset) — lower is better', fontsize=12)
ax.set_title(f'Failure Mode: Coreset Advantage Collapses on Anisotropic Data\n(m={SMALL_M})', fontsize=12)
ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=12)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3, axis='y')
for b in [*b1, *b2]:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.002, f'{b.get_height():.3f}',
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/task3_failure_mode_llh_gap.png', dpi=150)
plt.close()
print(f'Saved {RESULTS_DIR}/task3_failure_mode_llh_gap.png')

Saved results/task3_failure_mode_dataset.png
Saved results/task3_failure_mode_llh_gap.png


## Why the Method Fails: Explanation

On the spherical dataset, the coreset achieves a gap to full-data LLH of **0.004** while uniform sampling has a gap of **0.207** — the coreset is dramatically better. On the anisotropic dataset, the coreset's gap is **0.113** and uniform's is **0.209** — the coreset is still better than uniform, but its *advantage over uniform* shrinks from 0.20 to 0.10, a 2× reduction.

The failure mechanism is precisely Assumption 1: the rough cover B is built using Euclidean distance, which treats all directions equally. For an elongated cluster (e.g., std=10 in x, std=0.1 in y), two points at the far ends of the cluster are Euclidean-distant from B even though they are both well within the cluster's probability mass. This inflates their dist(x, B)² values, causing the coreset to over-sample points at the extremes of the elongated axis while under-sampling the narrow-axis directions where the cluster's density peaks.

Consequently, the coreset no longer uniformly approximates the log-likelihood φ(D|θ) — the guarantee of Theorem 3.1 weakens because the proof's reduction from GMM to k-means (Section 3.1) assumed that Euclidean distance faithfully represented the Gaussian structure, which fails here. The weighted EM run on this miscalibrated coreset finds a suboptimal solution.

This failure connects directly to Assumption 1 from Task 1.2: the ε-semi-spherical condition bounds the eigenvalue ratio precisely to prevent this Euclidean-distance miscalibration.

**Suggested modification:** Use a Mahalanobis distance in place of Euclidean distance when computing dist(x, B) in Algorithm 1 — estimating a per-cluster covariance matrix from B's Voronoi cells and using it to normalise distances within each cell. This would make the importance weights invariant to the cluster's orientation and aspect ratio, extending the guarantee to non-spherical Gaussians without requiring the semi-spherical assumption.